# Конвертация CLEAR examples на уровень commit/project

Одна запись результата соответствует одному `commit_hash`. Внутри записи находятся все CVE, CWE, файлы, функции и patch-изменения, относящиеся к этому commit.

```text
commit_hash
├── file 1
│   ├── function 1
│   └── function 2
└── file 2
```

Такая запись подходит для prompt, в котором LLM анализирует commit целиком.

In [ ]:
from pathlib import Path
import difflib
import json
import re
from collections import OrderedDict

## 1. Пути

Исходные advisory-файлы не изменяются. File-level результат также не изменяется.

In [ ]:
SOURCE_DIR = Path('/Users/nident/Desktop/JOB/ScolTech/CLEAR/dataset/examples')
OUTPUT_PATH = Path('/Users/nident/Desktop/JOB/ScolTech/CLEAR_Causal_Context_based_Agentic_Reasoning_for_Vulnerability_Detection2/dataset/java-pair/java_data_paired_examples_PROJECT_LVL.json')
SKIPPED_PATH = OUTPUT_PATH.with_name(OUTPUT_PATH.stem + '_skipped.json')
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
print('Источник:', SOURCE_DIR)
print('Результат:', OUTPUT_PATH)

## 2. Загрузка advisory-файлов

Каждый `update` будет добавлен в группу своего commit. Если один commit связан с несколькими CVE, они попадут в одну запись и будут перечислены в `cve_ids`.

In [ ]:
source_records = []
for path in sorted(SOURCE_DIR.glob('*.json')):
    source_records.append((path, json.loads(path.read_text(encoding='utf-8'))))
print('Advisory-файлов:', len(source_records))

## 3. Вспомогательные функции

`snippet` используется как function-level код. Если его нет, используется `context`. Каждый фрагмент получает заголовок с файлом и функцией, чтобы модель могла различать границы.

In [ ]:
def text(value):
    return value.strip() if isinstance(value, str) else ''

def code_from_side(side):
    if not isinstance(side, dict):
        return ''
    return text(side.get('snippet')) or text(side.get('context'))

def side_name(*sides):
    for side in sides:
        if isinstance(side, dict) and side.get('name'):
            return text(side['name'])
    return 'unknown_function'

def side_path(*sides):
    for side in sides:
        if isinstance(side, dict) and side.get('path'):
            return text(side['path'])
    return ''

def make_diff(before, after, file_name):
    return ''.join(difflib.unified_diff(
        before.splitlines(keepends=True), after.splitlines(keepends=True),
        fromfile=file_name + ' (before)', tofile=file_name + ' (after)'
    ))

## 4. Группировка по commit_hash

Ключом является только commit hash. Поэтому один commit может содержать несколько CVE и любое количество файлов. Повторяющиеся update-фрагменты удаляются по ключу advisory + update index + file + function.

In [ ]:
groups = OrderedDict()
seen_changes = set()
skipped = []

for source_path, record in source_records:
    cve_ids = record.get('cve_ids') or [record.get('id', '')]
    commits = record.get('commits') or []
    updates = record.get('updates') or []
    if not updates:
        skipped.append({'file': source_path.name, 'reason': 'no updates'})
        continue
    for update_index, update in enumerate(updates):
        before_side = update.get('before') or {}
        after_side = update.get('after') or {}
        before = code_from_side(before_side)
        after = code_from_side(after_side)
        file_name = side_path(after_side, before_side)
        commit_hash = update.get('commit') or (commits[-1] if commits else '')
        if not commit_hash or not before or not after or not file_name:
            skipped.append({'file': source_path.name, 'update': update_index, 'reason': 'missing commit, code or path'})
            continue
        function_name = side_name(after_side, before_side)
        change_key = (record.get('id'), update_index, file_name, function_name, commit_hash)
        if change_key in seen_changes:
            continue
        seen_changes.add(change_key)
        groups.setdefault(commit_hash, []).append({
            'source_advisory_id': record.get('id'),
            'cve_ids': cve_ids,
            'cwe_ids': record.get('cwe_ids') or ['CWE-Unknown'],
            'project': record.get('project', ''),
            'summary': record.get('summary', ''),
            'details': record.get('details', ''),
            'file_name': file_name,
            'function_name': function_name,
            'update_index': update_index,
            'before': before,
            'after': after,
            'diff': text(update.get('patch')) or make_diff(before, after, file_name)
        })
print('Commit-групп:', len(groups))
print('Пропущено updates:', len(skipped))

## 5. Сборка одной commit-level записи

В `func_before`, `func_after` и `func_diff` будет полный агрегированный prompt-контекст commit. Поля `files` и `changes` сохраняют структурированную копию.

In [ ]:
def aggregate_commit(commit_hash, changes, index):
    cve_ids = sorted({cve for change in changes for cve in change['cve_ids']})
    cwe_ids = sorted({cwe for change in changes for cwe in change['cwe_ids']})
    projects = sorted({change['project'] for change in changes if change['project']})
    before_parts, after_parts, diff_parts = [], [], []
    for change in changes:
        header = f"\n// ===== file: {change['file_name']} | function: {change['function_name']} | update: {change['update_index']} =====\n"
        before_parts.append(header + change['before'])
        after_parts.append(header + change['after'])
        diff_parts.append(header + change['diff'])
    files = {}
    for change in changes:
        files.setdefault(change['file_name'], []).append({
            'function_name': change['function_name'],
            'update_index': change['update_index'],
            'before': change['before'],
            'after': change['after'],
            'diff': change['diff']
        })
    return {
        'index': index,
        'commit_hash': commit_hash,
        'cve_id': ', '.join(cve_ids),
        'cwe_id': ', '.join(cwe_ids),
        'cve_ids': cve_ids,
        'cwe_ids': cwe_ids,
        'project': ', '.join(projects),
        'file_name': ', '.join(sorted(files)),
        'files': files,
        'func_before': '\n'.join(before_parts).strip(),
        'func_after': '\n'.join(after_parts).strip(),
        'func_diff': '\n'.join(diff_parts).strip(),
        'cve_description': '\n\n'.join(sorted({change['details'] or change['summary'] for change in changes})),
        'changes': changes
    }

converted = [aggregate_commit(commit, changes, i) for i, (commit, changes) in enumerate(groups.items())]
print('Итоговых commit-level записей:', len(converted))

## 6. Валидация

Основные поля сохраняются совместимыми с Java-Pair. Дополнительные поля позволяют построить project-level prompt без потери структуры.

In [ ]:
required = {'index', 'commit_hash', 'cve_id', 'cwe_id', 'file_name', 'func_before', 'func_after', 'func_diff', 'cve_description'}
errors = []
for i, row in enumerate(converted):
    missing = required - set(row)
    if missing:
        errors.append({'row': i, 'missing': sorted(missing)})
    if not row['commit_hash'] or not row['func_before'] or not row['func_after']:
        errors.append({'row': i, 'reason': 'empty commit or code'})
print('Ошибок:', len(errors))
assert not errors, errors[:5]
print(json.dumps({k: converted[0][k] for k in ('commit_hash', 'cve_ids', 'file_name', 'files')}, ensure_ascii=False, indent=2)[:5000])

In [ ]:
OUTPUT_PATH.write_text(json.dumps(converted, ensure_ascii=False, indent=2), encoding='utf-8')
SKIPPED_PATH.write_text(json.dumps(skipped, ensure_ascii=False, indent=2), encoding='utf-8')
print('Сохранено:', OUTPUT_PATH)
print('Отчёт о пропусках:', SKIPPED_PATH)

## 7. Запуск CLEAR

После выполнения ноутбука из корня CLEAR можно запускать `knowledge_data.py`. Он отправит модели один агрегированный prompt на один commit.

In [ ]:
# python knowledge_graph/knowledge_data.py \
#   --input dataset/java-pair/java_data_paired_examples_PROJECT_LVL.json \
#   --output knowledge_graph/java-pair/java_knowledge_data_PROJECT_LVL.json \
#   --llm_config llm_config.json \
#   --threads 8